# Inventory Stock Drift Analysis (Local Notebook Template)

> **CRITICAL PRIVACY DIRECTIVE**:
> - All data processing, statistical tests, and MLflow tracking in this notebook run **100% locally** on your machine.
> - Original Excel files must reside in `../data/private/` and are excluded from version control.
> - **Never commit this notebook with output cells populated with private data.** Always clear outputs (`Kernel -> Restart & Clear All Outputs`) before committing.

## 1. Imports and Local-Only MLflow Configuration

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

# 1. Strictly disable MLflow telemetry
os.environ["MLFLOW_ENABLE_TELEMETRY"] = "false"
os.environ["MLFLOW_DISABLE_TELEMETRY"] = "true"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "false"

import mlflow

# 2. Enforce local-only tracking URI inside project root
PROJECT_ROOT = Path("..").resolve()
LOCAL_MLFLOW_DIR = PROJECT_ROOT / "mlruns"
mlflow.set_tracking_uri(LOCAL_MLFLOW_DIR.as_uri())

print(f"Local MLflow Tracking URI: {mlflow.get_tracking_uri()}")
assert mlflow.get_tracking_uri().startswith("file:"), "Tracking URI must be local!"

## 2. Load Local Datasets (Excel / CSV)

Point to your private Excel files in `data/private/` or use the synthetic demonstration datasets in `data/`.

In [ ]:
# Option A: Private Excel files (Place your confidential data in data/private/)
ref_file = PROJECT_ROOT / "data" / "private" / "reference.xlsx"
cur_file = PROJECT_ROOT / "data" / "private" / "current.xlsx"

# Fallback Option B: Synthetic demonstration files (if private files not present)
if not ref_file.exists():
    ref_file = PROJECT_ROOT / "data" / "synthetic_demo_reference.csv"
    cur_file = PROJECT_ROOT / "data" / "synthetic_demo_current.csv"

def load_file(path):
    path = Path(path)
    if path.suffix.lower() in [".xlsx", ".xls", ".xlsm"]:
        return pd.read_excel(path)
    return pd.read_csv(path)

reference_df = load_file(ref_file)
current_df = load_file(cur_file)

print(f"Loaded Reference: {ref_file.name} ({len(reference_df)} rows)")
print(f"Loaded Current:   {cur_file.name} ({len(current_df)} rows)")

## 3. Identify Shared Numeric Features

In [ ]:
numeric_features = [
    col for col in reference_df.select_dtypes(include="number").columns
    if col in current_df.columns and col in current_df.select_dtypes(include="number").columns
]
print(f"Numeric features for drift testing: {numeric_features}")

## 4. Run Kolmogorov-Smirnov Drift Test & Log to Local MLflow

In [ ]:
threshold = 0.05
plots_dir = PROJECT_ROOT / "reports" / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

mlflow.set_experiment("Inventory_Stock_Drift_Analysis")
results = []

with mlflow.start_run():
    mlflow.log_param("reference_source", ref_file.name)
    mlflow.log_param("current_source", cur_file.name)
    mlflow.log_param("threshold", threshold)
    
    for feature in numeric_features:
        ref_vals = reference_df[feature].dropna()
        cur_vals = current_df[feature].dropna()
        
        if len(ref_vals) < 2 or len(cur_vals) < 2:
            continue
            
        stat, p_val = ks_2samp(ref_vals, cur_vals)
        drift_detected = bool(p_val < threshold)
        
        results.append({
            "feature": feature,
            "reference_mean": round(ref_vals.mean(), 4),
            "current_mean": round(cur_vals.mean(), 4),
            "ks_statistic": round(stat, 4),
            "p_value": round(p_val, 6),
            "drift_detected": drift_detected
        })
        
        # Log metrics locally
        mlflow.log_metric(f"{feature}_ks_stat", float(stat))
        mlflow.log_metric(f"{feature}_p_value", float(p_val))
        mlflow.log_metric(f"{feature}_drift", int(drift_detected))
        
        # Plot and save locally
        plt.figure(figsize=(7, 3))
        plt.hist(ref_vals, bins=20, alpha=0.6, label="Reference")
        plt.hist(cur_vals, bins=20, alpha=0.6, label="Current")
        plt.title(f"{feature} Distribution (Drift: {drift_detected})")
        plt.xlabel(feature)
        plt.ylabel("Count")
        plt.legend()
        plt.tight_layout()
        
        plot_path = plots_dir / f"{feature}_comparison.png"
        plt.savefig(plot_path, dpi=120)
        plt.close()
        mlflow.log_artifact(str(plot_path), artifact_path="plots")
        
    drift_report = pd.DataFrame(results)
    summary_csv = PROJECT_ROOT / "reports" / "drift_summary.csv"
    drift_report.to_csv(summary_csv, index=False)
    mlflow.log_artifact(str(summary_csv))

print("Analysis complete. Drift Summary Table:")
print(drift_report)

## 5. Review Locally in MLflow Dashboard

Launch the local MLflow UI in your terminal:
```bash
mlflow ui --host 127.0.0.1 --port 5000
```
Then visit `http://127.0.0.1:5000` in your web browser.